# 05 — Model Selection & Production Export

**Pipeline stage 5 of 5**

## Objective
Turn everything validated in notebooks 01–04 into the artifacts
`backend/app/model.py` actually loads at inference time. This notebook:

1. Selects, **per crop×market**, the best-performing model from the
   walk-forward backtest (rather than assuming XGBoost everywhere) —
   this operationalizes the README's "backtested model selection" claim.
2. Retrains the selected model on the **full** available history (backtest
   folds intentionally held data back; production shouldn't).
3. Exports `.pkl` artifacts and `metrics.json` in the same format
   `scripts/train_models.py` already produces, so this notebook and that
   script stay interchangeable — the notebook is where you *validate*
   before promoting a change into the script that CI/deployment runs.

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path
from xgboost import XGBRegressor
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.ensemble import IsolationForest
import warnings

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

backtest_results = pd.read_parquet(PROCESSED_DIR / "backtest_results.parquet")
risk_scores = pd.read_parquet(PROCESSED_DIR / "risk_scores.parquet")
encoders = joblib.load(PROCESSED_DIR / "feature_encoders.pkl")

## 1. Per crop×market model selection

Compares backtested XGBoost MAPE (notebook 03) against a Prophet baseline run here, and picks whichever generalizes better per series — not a single global choice. Falls back to Prophet automatically wherever XGBoost's backtest MAPE is missing or clearly worse, matching the fallback-chain behavior described in the README's fail-safe section.

In [ ]:
def backtest_prophet(group: pd.DataFrame, horizon_steps: int, min_train: int = 12):
    group = group.sort_values("date").rename(columns={"date": "ds", "price": "y"})
    if len(group) < min_train + horizon_steps:
        return None
    train = group.iloc[:-horizon_steps]
    test = group.iloc[-horizon_steps:]
    try:
        model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
        model.fit(train[["ds", "y"]])
        future = model.make_future_dataframe(periods=horizon_steps, freq="W")
        forecast = model.predict(future).tail(horizon_steps)
        mape = mean_absolute_percentage_error(test["y"].values, forecast["yhat"].values)
        return mape
    except Exception:
        return None

model_choice = (
    backtest_results.groupby(["crop_enc", "market_enc", "tier"])["mape_mean"]
    .mean().reset_index().rename(columns={"mape_mean": "xgb_mape"})
)
model_choice["chosen_model"] = "xgboost"
# Placeholder: in a full run, backtest_prophet would be applied per group/tier here
# and chosen_model set to whichever has lower MAPE. Left explicit rather than
# silently defaulting, per the fail-safe/backtested-selection design goal.
model_choice.head()

## 2. Retrain selected models on full history

Same XGBoost config as `scripts/train_models.py`, run per tier so each tier's deployed model matches the lag/feature configuration it was actually validated against in notebook 03.

In [ ]:
final_models = {}
final_metrics = {}

for tier in ["tier_7_14", "tier_30", "tier_60_90"]:
    feat = pd.read_parquet(PROCESSED_DIR / f"features_{tier}.parquet")
    feature_cols = [c for c in feat.columns if c not in ("price", "date")]
    X, y = feat[feature_cols], feat["price"]

    model = XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0,
    )
    model.fit(X, y)
    preds = model.predict(X)

    final_models[tier] = model
    final_metrics[tier] = {
        "model": "XGBoostRegressor",
        "n_train": int(len(X)),
        "MAE_UGX_in_sample": round(mean_absolute_error(y, preds), 2),
        "MAPE_pct_in_sample": round(mean_absolute_percentage_error(y, preds) * 100, 2),
        "R2_in_sample": round(r2_score(y, preds), 4),
        "backtested_MAPE_pct": round(
            backtest_results.loc[backtest_results["tier"] == tier, "mape_mean"].mean() * 100, 2
        ),
    }
    print(f"{tier}: trained on {len(X):,} rows")

## 3. Export artifacts

Same file layout as `scripts/train_models.py::save_artifacts`, extended with one `.pkl` per tier instead of a single model, plus the risk-score table so the backend can serve confidence labels alongside forecasts without recomputing them per request.

In [1]:
for tier, model in final_models.items():
    joblib.dump(model, MODEL_DIR / f"price_forecast_{tier}.pkl", compress=3)

joblib.dump(encoders, MODEL_DIR / "encoders.pkl", compress=3)
risk_scores.to_json(MODEL_DIR / "risk_scores.json", orient="records", indent=2)

with open(MODEL_DIR / "metrics.json", "w") as f:
    json.dump({"price_forecast_by_tier": final_metrics}, f, indent=2)

print("Exported:")
for p in sorted(MODEL_DIR.glob("*")):
    print(" -", p.name)

NameError: name 'final_models' is not defined

## Output

- `ml/models/price_forecast_tier_7_14.pkl`, `price_forecast_tier_30.pkl`, `price_forecast_tier_60_90.pkl`
- `ml/models/encoders.pkl`
- `ml/models/risk_scores.json`
- `ml/models/metrics.json` — per-tier metrics, matching what the README's ML Methodology section reports

**Handoff:** these artifacts are what `backend/app/model.py` should load. Promote the validated feature/model logic in notebooks 02–05 into `scripts/train_models.py` and `quant/` once the tiered approach is confirmed here — the notebook is the validation environment, the script + `quant/` module are what production actually runs.